# Talks markdown generator for academicpages

Takes a TSV of talks with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `talks.py`. Run either from the `markdown_generator` folder after replacing `talks.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases, rather than Stuart's non-standard TSV format and citation style.

In [20]:
import pandas as pd
import os

## Data format

The TSV needs to have the following columns: title, type, url_slug, venue, date, location, talk_url, description, with a header at the top. Many of these fields can be blank, but the columns must be in the TSV.

- Fields that cannot be blank: `title`, `url_slug`, `date`. All else can be blank. `type` defaults to "Talk" 
- `date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. 
    - The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/talks/YYYY-MM-DD-[url_slug]`
    - The combination of `url_slug` and `date` must be unique, as it will be the basis for your filenames

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [22]:
!cat talks.tsv

title	type	url_slug	venue	date	location	talk_url	description
"Unveiling the Soft X-ray Source Population of the Inner Galactic Disk with XMM-Newton"	Talk	soft-xray-population-CLAP Workshop	Colma di Sormano	2025-09-10	Como, Italy		Soft X-ray in the inner Galactic Disk using XMM heritage survey
"Exploring the origin and evolution of close binaries in Globular clusters and the Galactic Center"	Talk	close-binaries-star-cluster	Structure, Formation, and Evolution of Star Clusters	2025-03-30	Dali, China		Formation and evolution mechanisms of CVs in globular clusters and galactic centers
"Periodic X-ray sources: diagnosing the nature and evolution for exotic binaries in dense environment"	Talk	periodic-xray-sources-columbia	Department of Physics, Columbia University	2023-04-01	New York, USA		This talk covers periodic X-ray sources and their dynamical evolution in dense stellar environments.
"Diagnosing dynamical formation and evolution of cataclysmic variables in globular clusters"	Talk	star-

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [23]:
talks = pd.read_csv("talks.tsv", sep="\t", header=0)
talks

,title,type,url_slug,venue,date,location,talk_url,description
0,Unveiling the Soft X-ray Source Population of ...,Talk,soft-xray-population-CLAP Workshop,Colma di Sormano,2025-09-10,"Como, Italy",NaN,Soft X-ray in the inner Galactic Disk using XM...
1,Exploring the origin and evolution of close bi...,Talk,close-binaries-star-cluster,"Structure, Formation, and Evolution of Star Cl...",2025-03-30,"Dali, China",NaN,Formation and evolution mechanisms of CVs in g...
2,Periodic X-ray sources: diagnosing the nature ...,Talk,periodic-xray-sources-columbia,"Department of Physics, Columbia University",2023-04-01,"New York, USA",NaN,This talk covers periodic X-ray sources and th...
3,Diagnosing dynamical formation and evolution o...,Talk,star-clusters-cv-dynamics,"Structure, Formation, and Evolution of Star Cl...",2023-04-23,"Zhuhai, China",NaN,Discussing formation and evolution mechanisms ...
4,Periodic X-ray sources in globular clusters: d...,Poster,periodic-xray-sources-HEAD,Hawaii,2023-03-23,"Hawaii, USA",NaN,This Poster covers periodic X-ray sources and ...
5,Exploring the Outskirts of Globular Clusters w...,Talk,erossita-globular-outskirts,eROSITA Scientific Exploitation Workshop,2023-01-01,"Suzhou, China",NaN,This talk presents results from the eROSITA su...
6,Searching for Quasi-periodic Oscillations in A...,Talk,research-agn-variations,Research on the Variations of Active Galactic ...,2023-03-17,"Xiamen, China",NaN,Focusing on QPOs in AGN using Chandra Deep Fie...
7,Searching for Quasi-periodic Oscillations in A...,Talk,xray-astronomy-60th-anniversary,The 60th Anniversary of X-Ray Astronomy,2022-06-17,"Beijing, China",NaN,Further discussion on QPOs in AGN for the Chan...
8,Periodic X-ray Sources in the Galactic Center/...,Talk,jiangsu-astronomical-society-2021,2021-2022 Academic Annual Meeting of Jiangsu A...,2022-05-10,"Nanjing, China",NaN,Analysis of periodic X-ray sources in Galactic...
9,Searching for Quasi-periodic Oscillations of A...,Talk,chandra-data-science-2021,Chandra Data Science: Novel Methods in Computi...,2021-04-21,Online,NaN,Presentation on QPOs of AGN using advanced com...


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [24]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    if type(text) is str:
        return "".join(html_escape_table.get(c,c) for c in text)
    else:
        return "False"

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [27]:
loc_dict = {}

for row, item in talks.iterrows():
    
    md_filename = str(item.date) + "-" + item.url_slug + ".md"
    html_filename = str(item.date) + "-" + item.url_slug 
    year = item.date[:4]
    
    md = "---\ntitle: \""   + item.title + '"\n'
    md += "collection: talks" + "\n"
    
    if len(str(item.type)) > 3:
        md += 'type: "' + item.type + '"\n'
    else:
        md += 'type: "Talk"\n'
    
    md += "permalink: /talks/" + html_filename + "\n"
    
    if len(str(item.venue)) > 3:
        md += 'venue: "' + item.venue + '"\n'
        
    if len(str(item.location)) > 3:
        md += "date: " + str(item.date) + "\n"
    
    if len(str(item.location)) > 3:
        md += 'location: "' + str(item.location) + '"\n'
           
    md += "---\n"
    
    
    if len(str(item.talk_url)) > 3:
        md += "\n[More information here](" + item.talk_url + ")\n" 
        
    
    if len(str(item.description)) > 3:
        md += "\n" + html_escape(item.description) + "\n"
        
        
    md_filename = os.path.basename(md_filename)
    #print(md)
    
    with open("../_talks/" + md_filename, 'w') as f:
        f.write(md)

These files are in the talks directory, one directory below where we're working from.

In [28]:
!ls ../_talks

2020-01-01-xray-binary-symposium-2020.md
2020-10-08-chandra-frontiers-time-domain-2020.md
2021-01-12-xray-binary-symposium-2021.md
2021-04-21-chandra-data-science-2021.md
2022-05-10-jiangsu-astronomical-society-2021.md
2022-06-17-xray-astronomy-60th-anniversary.md
2023-01-01-erossita-globular-outskirts.md
2023-03-17-research-agn-variations.md
2023-03-23-periodic-xray-sources-HEAD.md
2023-04-01-periodic-xray-sources-columbia.md
2023-04-23-star-clusters-cv-dynamics.md
2025-03-30-close-binaries-star-cluster.md
2025-09-10-soft-xray-population-CLAP Workshop.md


In [19]:
!cat ../_talks/2013-03-01-tutorial-1.md

cat: ../_talks/2013-03-01-tutorial-1.md: No such file or directory
